# Verify video pipeline

Confirm a raw LRS3-trainval frame, after the grayscale transform the
frozen appearance encoder expects (grayscale is applied at data-LOADING
time, not baked into the saved crop files), ends up single-channel and `(96, 96)`.

**What "looks right" means:**
- After grayscale + resize/crop, a trainval frame's tensor shape is
  `(1, 96, 96)`.
- dtype and value range (e.g. `uint8` 0-255) are what the appearance
  encoder's transform pipeline expects.
- The frame looks like a reasonable mouth crop -- see the TODO on the
  transform below: the placeholder resize keeps far more of the face than
  a real mouth crop would.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import torchvision
import torchvision.transforms as T
from datasets import load_dataset

## Config

In [ ]:
# TODO: fill in a real path from notebook 01's lrs3_trainval manifest video_path column
import random, pandas as pd

lrs3_trainval_manifest = pd.read_csv("lrs3_trainval_manifest.csv")
lrs3_sample = lrs3_trainval_manifest.iloc[int(random.uniform(0, len(lrs3_trainval_manifest)))]
SAMPLE_TRAINVAL_VIDEO_PATH = Path(lrs3_sample["video_path"])

## Load one raw LRS3-trainval frame

In [3]:
from scripts.extract_landmarks_grid import _read_video_via_torchcodec
video_frames, _audio, _info = _read_video_via_torchcodec(str(SAMPLE_TRAINVAL_VIDEO_PATH), pts_unit="sec")
raw_frame = video_frames[0]  # (H, W, C), uint8, RGB
print(f"raw frame shape: {tuple(raw_frame.shape)}, dtype: {raw_frame.dtype}")

raw frame shape: (224, 224, 3), dtype: torch.uint8


## Apply the grayscale + resize/crop transform

TODO: this should match whatever `fusion_avsr`'s data-loading transform
pipeline eventually implements for the appearance encoder (mirroring
auto-AVSR's `transforms.py`) -- I'll fill in the real crop/resize logic once
that exists; the `Grayscale()` + `Resize((96, 96))` below is a
placeholder standing in for it.

In [5]:
transform = T.Compose([
    T.ToPILImage(),
    T.Grayscale(),
    T.Resize((96, 96)),  # TODO: I'll replace with the real mouth-crop logic, not a naive resize
    T.PILToTensor(),
])

# torchvision frames are (H, W, C); ToPILImage expects (C, H, W)
processed_frame = transform(raw_frame.permute(2, 0, 1))
print(f"processed frame shape: {tuple(processed_frame.shape)}, dtype: {processed_frame.dtype}")

processed frame shape: (1, 96, 96), dtype: torch.uint8


## Visual check

In [ ]:
plt.imshow(processed_frame.squeeze(0), cmap="gray")
plt.title("processed trainval crop")
plt.axis("off")
plt.show()

## TODO checklist

- [ ] `processed_frame.shape == (1, 96, 96)`.
- [ ] The frame is single-channel (grayscale).
- [ ] Crop framing is a tight mouth crop (not the whole face) once the
      real mouth-crop logic replaces the placeholder resize.